# 00 — Data Acquisition

This notebook does three things:

1. **Adds new proteins to the dataset** — you specify a protein name and UniProt ID,
   and the notebook automatically finds the best experimental structures on RCSB,
   downloads them, and updates `proteins.csv`.
2. **Backfills missing experimental CIF files** for proteins already in the manifest
   (useful when setting up on a new machine or after the manifest is updated).
3. **Downloads AlphaFold2 and OpenFold3 predictions** for every protein in the manifest.

**Notebooks run order:** `00 → 01 → 02 → 03`

**Before running:** add your NIM API key to a `.env` file at the repo root:
```
NIM_API_KEY=nvapi-xxxxxxxxxxxxxxxxxxxx
```
Keys are available at https://build.nvidia.com/openfold/openfold3


## Setup

In [1]:
import sys, os, json
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

from src.downloader import (
    search_rcsb_by_uniprot,
    download_rcsb,
    download_alphafold2,
    fetch_uniprot_sequence,
    predict_openfold3,
)
from src.downloader import load_proteins

load_dotenv()

nim_key_loaded = bool(os.environ.get("NIM_API_KEY"))
print(f"NIM_API_KEY found: {nim_key_loaded}")
if not nim_key_loaded:
    print("  Add NIM_API_KEY=nvapi-... to a .env file at the repo root.")

NIM_API_KEY found: True


In [2]:
DATA_DIR = "data"
EXP_DIR = os.path.join(DATA_DIR, "experimental")
AF2_DIR = os.path.join(DATA_DIR, "alphafold2")
OF3_DIR = os.path.join(DATA_DIR, "openfold3")

for d in [EXP_DIR, AF2_DIR, OF3_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}")

proteins = pd.read_csv("proteins.csv")
print(f"\n{len(proteins)} proteins currently in manifest:")
display(proteins[["protein_name", "uniprot_id", "pdb_xray", "pdb_nmr", "pdb_cryoem"]])

  data/experimental
  data/alphafold2
  data/openfold3

73 proteins currently in manifest:


,protein_name,uniprot_id,pdb_xray,pdb_nmr,pdb_cryoem
0,Barnase,P00648,6PQK,1BNR,NaN
1,Protein G (GB1 domain parent),P06654,1IGD,1GB1,9G4T
2,Adenylate kinase (E. coli),P69441,7APU,NaN,NaN
3,T4 lysozyme,P00720,2O7A,2LC9,8DCS
4,Myoglobin,P02185,5YCE,1MYF,NaN
...,...,...,...,...,...
68,Nucleoporin Nup133,Q8WUM0,1XKS,NaN,NaN
69,Importin beta-1,Q14974,1IBR,NaN,9N85
70,V-ATPase subunit A,P38606,NaN,NaN,6WLZ
71,Photosystem II protein D1,P0C0S1,5AJI,NaN,9E68


---
## Add New Proteins

Edit the cell below to add the proteins you want to include.
Each entry needs a human-readable name and a UniProt accession.

The notebook will:
1. Query RCSB for the best X-ray, NMR, and Cryo-EM structure for each protein
2. Show you a review table of what was found
3. Download the selected structures
4. Append the new rows to `proteins.csv`

Proteins already in `proteins.csv` are automatically skipped.

**Resolution cutoff:** X-ray and Cryo-EM structures worse than `RESOLUTION_CUTOFF` Å are excluded (configurable below).


In [3]:
NEW_PROTEINS = load_proteins("data/new_proteins.txt")

# Maximum number of candidate structures returned per method.
MAX_CANDIDATES_PER_METHOD = 50

# Resolution cutoff in Angstroms for X-ray and Cryo-EM.
# Structures worse than this are excluded.
RESOLUTION_CUTOFF = 3.5

### Step 1 — Check which proteins are new


In [4]:
existing_uniprots = set(proteins["uniprot_id"].str.strip().tolist())

to_add = []
skipped = []

for name, uid in NEW_PROTEINS:
    if uid.strip() in existing_uniprots:
        skipped.append((name, uid))
    else:
        to_add.append((name, uid.strip()))

if skipped:
    print(f"Already in proteins.csv (skipping):")
    for name, uid in skipped:
        print(f"  {name} ({uid})")

if not to_add:
    print("\nNo new proteins to add. Edit NEW_PROTEINS above and re-run.")
else:
    print(f"\n{len(to_add)} new protein(s) to process:")
    for name, uid in to_add:
        print(f"  {name} ({uid})")

Already in proteins.csv (skipping):
  Barnase (P00648)
  Protein G (GB1 domain parent) (P06654)
  Adenylate kinase (E. coli) (P69441)
  T4 lysozyme (P00720)
  Myoglobin (P02185)
  Lysozyme C (human) (P61626)
  Ribonuclease A (P61823)
  BPTI (aprotinin) (P00974)
  HIV-1 protease (P03366)
  Dihydrofolate reductase (E. coli) (P0ABQ4)
  Thioredoxin (E. coli) (P0AA25)
  CheY response regulator (P0AE67)
  HRas GTPase (P01112)
  Calmodulin (P0DP23)
  Cyclophilin A (P62937)
  FKBP12 (P62942)
  Green fluorescent protein (P42212)
  Staphylococcal nuclease (P00644)
  TEM-1 beta-lactamase (P62593)
  Cytochrome c (P99999)
  Carbonic anhydrase II (human) (P00918)
  Superoxide dismutase 1 (human) (P00441)
  Transthyretin (P02766)
  Beta-2-microglobulin (P61769)
  Tumor protein p53 (P04637)
  Actin (beta) (P60709)
  Tubulin beta chain (P07437)
  Hsp90-alpha (P07900)
  GroEL chaperonin (P0A6F5)
  ClpP protease (P0A7K2)
  RNA polymerase II subunit RPB1 (P24928)
  DNA-dependent protein kinase catalytic s

### Step 2 — Search RCSB for experimental structures


In [5]:
search_results = {}
for name, uid in to_add:
    print(f"\n{'='*55}")
    print(f"  Searching RCSB for: {name} ({uid})")
    print(f"{'='*55}")
    try:
        hits = search_rcsb_by_uniprot(
            uid,
            methods=("X-ray", "NMR", "Cryo-EM"),
            max_per_method=MAX_CANDIDATES_PER_METHOD,
            resolution_cutoff=RESOLUTION_CUTOFF,
        )
        search_results[uid] = {"name": name, "hits": hits}
    except Exception as exc:
        print(f"  [ERROR] RCSB search failed for {uid}: {exc}")
        search_results[uid] = {"name": name, "hits": {}}


  Searching RCSB for: Ubiquitin (P62988)
  [warn] No RCSB entries found for UniProt ID P62988

  Searching RCSB for: Elastin (P15502)
  [warn] No RCSB entries found for UniProt ID P15502

  Searching RCSB for: Aspartate transcarbamoylase (P0A7F9)
  [warn] No RCSB entries found for UniProt ID P0A7F9


### Step 3 — Review selected structures

The table below shows the **best structure chosen per method** (lowest resolution for
X-ray/Cryo-EM; first deposited for NMR). Review it carefully before proceeding.

If you want to override a selection, edit the `OVERRIDES` dict in the next cell.


In [6]:
# Build the best-pick selection from search results
selections = {}

for uid, data in search_results.items():
    name = data["name"]
    hits = data["hits"]

    def best(method):
        bucket = hits.get(method, [])
        return bucket[0] if bucket else None

    xray = best("X-ray")
    nmr = best("NMR")
    cryoem = best("Cryo-EM")

    # Prefer chain from X-ray; fall back to NMR or Cryo-EM
    chain = (xray or nmr or cryoem or {}).get("chain_id", "A")

    selections[uid] = {
        "protein_name": name,
        "uniprot_id": uid,
        "pdb_xray": xray["pdb_id"] if xray else "",
        "res_xray": xray["resolution"] if xray else None,
        "pdb_nmr": nmr["pdb_id"] if nmr else "",
        "pdb_cryoem": cryoem["pdb_id"] if cryoem else "",
        "res_cryoem": cryoem["resolution"] if cryoem else None,
        "chain_id": chain,
    }

# Print review table
if selections:
    rows = []
    for uid, s in selections.items():
        rows.append(
            {
                "protein": s["protein_name"],
                "uniprot": uid,
                "X-ray PDB": s["pdb_xray"] or "(none)",
                "X-ray res": f"{s['res_xray']:.2f} Å" if s["res_xray"] else "—",
                "NMR PDB": s["pdb_nmr"] or "(none)",
                "Cryo-EM PDB": s["pdb_cryoem"] or "(none)",
                "Cryo-EM res": f"{s['res_cryoem']:.2f} Å" if s["res_cryoem"] else "—",
                "chain": s["chain_id"],
            }
        )
    display(pd.DataFrame(rows).set_index("protein"))
else:
    print("Nothing to show — add proteins to NEW_PROTEINS above.")

,uniprot,X-ray PDB,X-ray res,NMR PDB,Cryo-EM PDB,Cryo-EM res,chain
protein,,,,,,,
Ubiquitin,P62988,(none),—,(none),(none),—,A
Elastin,P15502,(none),—,(none),(none),—,A
Aspartate transcarbamoylase,P0A7F9,(none),—,(none),(none),—,A


#### Override selections (optional)

If you want to use a different PDB ID than the one found automatically, edit the
`OVERRIDES` dict below. Leave it empty `{}` to use the automatic selections.

```python
# Example:
OVERRIDES = {
    "P62988": {           # UniProt ID of the protein to override
        "pdb_xray":   "1UBQ",    # override X-ray (or leave key out to keep auto)
        "pdb_nmr":    "1D3Z",    # override NMR
        "pdb_cryoem": "",        # explicitly leave empty
        "chain_id":   "A",       # override chain
    },
}
```


In [7]:
# ── EDIT IF NEEDED ────────────────────────────────────────────────────────────
OVERRIDES = {
    # "UniProtID": {"pdb_xray": "XXXX", "pdb_nmr": "XXXX", "pdb_cryoem": "XXXX", "chain_id": "A"},
}

# Apply overrides to selections
for uid, overrides in OVERRIDES.items():
    if uid not in selections:
        print(
            f"[WARN] Override for {uid} but that protein is not in selections — ignored."
        )
        continue
    for field, value in overrides.items():
        if field in selections[uid]:
            old = selections[uid][field]
            selections[uid][field] = value
            print(f"  Override {uid} {field}: {old!r} → {value!r}")

print("Overrides applied." if OVERRIDES else "No overrides defined.")

No overrides defined.


### Step 4 — Download experimental structures


In [8]:
download_log = []

for uid, s in selections.items():
    name = s["protein_name"]
    print(f"\n{'='*55}")
    print(f"  {name}  ({uid})")
    print(f"{'='*55}")

    for method, pdb_id in [
        ("X-ray", s["pdb_xray"]),
        ("NMR", s["pdb_nmr"]),
        ("Cryo-EM", s["pdb_cryoem"]),
    ]:
        if not pdb_id or pdb_id.strip().lower() in ("", "nan", "none"):
            print(f"  [{method:8s}] not available — skip")
            continue
        try:
            path = download_rcsb(pdb_id.upper(), EXP_DIR)
            download_log.append(
                {"protein": name, "method": method, "pdb_id": pdb_id, "status": "ok"}
            )
        except Exception as exc:
            print(f"  [{method:8s}] ERROR: {exc}")
            download_log.append(
                {
                    "protein": name,
                    "method": method,
                    "pdb_id": pdb_id,
                    "status": str(exc),
                }
            )

if download_log:
    print("\nDownload summary:")
    display(pd.DataFrame(download_log))
else:
    print("Nothing downloaded.")


  Ubiquitin  (P62988)
  [X-ray   ] not available — skip
  [NMR     ] not available — skip
  [Cryo-EM ] not available — skip

  Elastin  (P15502)
  [X-ray   ] not available — skip
  [NMR     ] not available — skip
  [Cryo-EM ] not available — skip

  Aspartate transcarbamoylase  (P0A7F9)
  [X-ray   ] not available — skip
  [NMR     ] not available — skip
  [Cryo-EM ] not available — skip
Nothing downloaded.


### Step 5 — Update proteins.csv

This cell appends the new proteins to `proteins.csv`. It only writes rows for
proteins whose experimental download succeeded (at least one CIF downloaded).

Review the preview before uncommenting the write line.


In [9]:
new_rows = []

for uid, s in selections.items():
    # Only add if at least one experimental structure was downloaded
    downloaded_ok = any(
        log["status"] == "ok" and log["protein"] == s["protein_name"]
        for log in download_log
    )
    if not downloaded_ok:
        print(
            f"  [skip] {s['protein_name']}: no experimental structures downloaded successfully"
        )
        continue

    # Determine NMR model (default 0; blank for proteins without NMR)
    nmr_model = "0" if s["pdb_nmr"] else ""

    new_rows.append(
        {
            "protein_name": s["protein_name"],
            "uniprot_id": uid,
            "pdb_xray": s["pdb_xray"],
            "pdb_nmr": s["pdb_nmr"],
            "pdb_cryoem": s["pdb_cryoem"],
            "chain_id": s["chain_id"],
            "nmr_model": nmr_model,
            "notes": f"Added automatically via RCSB search. "
            f"X-ray res: {s.get('res_xray') or 'n/a'}. "
            f"Cryo-EM res: {s.get('res_cryoem') or 'n/a'}. "
            f"Review chain_id and nmr_model before running pipeline.",
        }
    )

if not new_rows:
    print("No new rows to add.")
else:
    preview_df = pd.DataFrame(new_rows)
    print(f"Preview — {len(new_rows)} row(s) to append to proteins.csv:")
    display(
        preview_df[
            [
                "protein_name",
                "uniprot_id",
                "pdb_xray",
                "pdb_nmr",
                "pdb_cryoem",
                "chain_id",
            ]
        ]
    )
    print()

    # Reload the CSV fresh (in case it was edited elsewhere), then append.
    proteins_current = pd.read_csv("proteins.csv")
    proteins_updated = pd.concat([proteins_current, preview_df], ignore_index=True)
    proteins_updated.to_csv("proteins.csv", index=False)
    proteins = proteins_updated  # update the in-memory reference for cells below
    print(f"proteins.csv updated — {len(proteins_updated)} total proteins.")
    display(
        proteins_updated[
            ["protein_name", "uniprot_id", "pdb_xray", "pdb_nmr", "pdb_cryoem"]
        ]
    )

  [skip] Ubiquitin: no experimental structures downloaded successfully
  [skip] Elastin: no experimental structures downloaded successfully
  [skip] Aspartate transcarbamoylase: no experimental structures downloaded successfully
No new rows to add.


---
## Backfill missing experimental structures

This section ensures every protein already in `proteins.csv` has its
experimental CIF files on disk — including proteins that were added in a
previous session or on a different machine.

For each protein in the manifest, it checks whether the X-ray, NMR, and
Cryo-EM files are present and downloads any that are missing.
Proteins with no PDB ID for a given method are skipped automatically.
Files that already exist are not re-downloaded.


In [10]:
# Reload manifest (picks up any rows added in the Add New Proteins section above)
proteins = pd.read_csv("proteins.csv")

backfill_log = []

for _, row in proteins.iterrows():
    name = row["protein_name"]
    uid = row["uniprot_id"]

    for method, col in [
        ("X-ray", "pdb_xray"),
        ("NMR", "pdb_nmr"),
        ("Cryo-EM", "pdb_cryoem"),
    ]:
        pdb_id = row.get(col)
        if pd.isna(pdb_id) or not str(pdb_id).strip():
            continue  # no structure for this method

        pdb_id = str(pdb_id).strip().upper()
        dest = os.path.join(EXP_DIR, f"{pdb_id}.cif")

        if os.path.exists(dest):
            continue  # already on disk

        print(f"  [{method:8s}] {name} — downloading {pdb_id}")
        try:
            download_rcsb(pdb_id, EXP_DIR)
            backfill_log.append(
                {"protein": name, "method": method, "pdb_id": pdb_id, "status": "ok"}
            )
        except Exception as exc:
            print(f"  [ERROR] {pdb_id}: {exc}")
            backfill_log.append(
                {
                    "protein": name,
                    "method": method,
                    "pdb_id": pdb_id,
                    "status": str(exc),
                }
            )

if backfill_log:
    print(f"\nBackfill complete — {len(backfill_log)} file(s) downloaded or attempted.")
    display(pd.DataFrame(backfill_log))
else:
    print("All experimental CIF files already present — nothing to download.")

All experimental CIF files already present — nothing to download.


---
## AlphaFold2 predictions

Downloads AF2 predictions from the AlphaFold Protein Structure Database
(https://alphafold.ebi.ac.uk/) for **every protein currently in `proteins.csv`**.
The API always returns the latest model version. Existing files are skipped.


In [11]:
# Reload proteins.csv to pick up any new rows written above
proteins = pd.read_csv("proteins.csv")
print(f"Downloading AF2 for {len(proteins)} proteins...")

for _, row in proteins.iterrows():
    print(f"  {row['protein_name']} ({row['uniprot_id']})")
    try:
        download_alphafold2(row["uniprot_id"], AF2_DIR)
    except Exception as exc:
        print(f"    [ERROR] {exc}")

  Barnase (P00648)
  [skip] AF2_P00648.cif already exists
  Protein G (GB1 domain parent) (P06654)
  [skip] AF2_P06654.cif already exists
  Adenylate kinase (E. coli) (P69441)
  [skip] AF2_P69441.cif already exists
  T4 lysozyme (P00720)
    [ERROR] 404 Client Error: Not Found for url: https://alphafold.ebi.ac.uk/api/prediction/P00720
  Myoglobin (P02185)
  [skip] AF2_P02185.cif already exists
  Lysozyme C (human) (P61626)
  [skip] AF2_P61626.cif already exists
  Ribonuclease A (P61823)
  [skip] AF2_P61823.cif already exists
  BPTI (aprotinin) (P00974)
  [skip] AF2_P00974.cif already exists
  HIV-1 protease (P03366)
    [ERROR] 404 Client Error: Not Found for url: https://alphafold.ebi.ac.uk/api/prediction/P03366
  Dihydrofolate reductase (E. coli) (P0ABQ4)
  [skip] AF2_P0ABQ4.cif already exists
  Thioredoxin (E. coli) (P0AA25)
  [skip] AF2_P0AA25.cif already exists
  CheY response regulator (P0AE67)
  [skip] AF2_P0AE67.cif already exists
  HRas GTPase (P01112)
  [skip] AF2_P01112.cif 

---
## OpenFold3 predictions (NVIDIA NIM)

Fetches the canonical UniProt sequence for each protein and submits it to the
OpenFold3 endpoint on NVIDIA NIM. Results are saved as `data/openfold3/OF3_<UniProt_ID>.cif`.

**Requires `NIM_API_KEY`** in your `.env` file.
If the key is missing, this section will print an error and skip — everything else still works.

Model reference: Wohlwend et al., Nature 2024 — https://www.nature.com/articles/s41586-024-07487-w


In [12]:
# Reload proteins.csv to pick up any new rows
proteins = pd.read_csv("proteins.csv")

sequences = {}

for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    try:
        seq = fetch_uniprot_sequence(uid)
        sequences[uid] = seq
        print(f"  {row['protein_name']} ({uid}): {len(seq)} aa")
    except Exception as exc:
        print(f"  [ERROR] {row['protein_name']} ({uid}): {exc}")

print(f"\n{len(sequences)} / {len(proteins)} sequences fetched")

  Barnase (P00648): 157 aa
  Protein G (GB1 domain parent) (P06654): 448 aa
  Adenylate kinase (E. coli) (P69441): 214 aa
  T4 lysozyme (P00720): 164 aa
  Myoglobin (P02185): 154 aa
  Lysozyme C (human) (P61626): 148 aa
  Ribonuclease A (P61823): 150 aa
  BPTI (aprotinin) (P00974): 100 aa
  HIV-1 protease (P03366): 1447 aa
  Dihydrofolate reductase (E. coli) (P0ABQ4): 159 aa
  Thioredoxin (E. coli) (P0AA25): 109 aa
  CheY response regulator (P0AE67): 129 aa
  HRas GTPase (P01112): 189 aa
  Calmodulin (P0DP23): 149 aa
  Cyclophilin A (P62937): 165 aa
  FKBP12 (P62942): 108 aa
  Green fluorescent protein (P42212): 238 aa
  Staphylococcal nuclease (P00644): 231 aa
  TEM-1 beta-lactamase (P62593): 286 aa
  Cytochrome c (P99999): 105 aa
  Carbonic anhydrase II (human) (P00918): 260 aa
  Superoxide dismutase 1 (human) (P00441): 154 aa
  Transthyretin (P02766): 147 aa
  Beta-2-microglobulin (P61769): 119 aa
  Tumor protein p53 (P04637): 393 aa
  Actin (beta) (P60709): 375 aa
  Tubulin beta ch

In [13]:
of3_log = []
total = len(proteins)

for idx, (_, row) in enumerate(proteins.iterrows(), start=1):
    uid = row["uniprot_id"]
    name = row["protein_name"]
    seq = sequences.get(uid)

    print(f"[{idx}/{total}] {name} ({uid})", end=" ")

    if seq is None:
        print("— skipped (no sequence)")
        of3_log.append(
            {"protein_name": name, "uniprot_id": uid, "path": None, "status": "skipped"}
        )
        continue

    print(f"— {len(seq)} aa")
    try:
        path = predict_openfold3(
            uniprot_id=uid,
            sequence=seq,
            save_dir=OF3_DIR,
            diffusion_samples=1,
            timeout=300,
        )
        of3_log.append(
            {"protein_name": name, "uniprot_id": uid, "path": path, "status": "ok"}
        )
    except Exception as exc:
        print(f"  [ERROR] {exc}")
        of3_log.append(
            {"protein_name": name, "uniprot_id": uid, "path": None, "status": str(exc)}
        )

if of3_log:
    display(pd.DataFrame(of3_log))

[1/73] Barnase (P00648) — 157 aa
  [skip] OF3_P00648.cif already exists
[2/73] Protein G (GB1 domain parent) (P06654) — 448 aa
  [skip] OF3_P06654.cif already exists
[3/73] Adenylate kinase (E. coli) (P69441) — 214 aa
  [skip] OF3_P69441.cif already exists
[4/73] T4 lysozyme (P00720) — 164 aa
  [skip] OF3_P00720.cif already exists
[5/73] Myoglobin (P02185) — 154 aa
  [skip] OF3_P02185.cif already exists
[6/73] Lysozyme C (human) (P61626) — 148 aa
  [skip] OF3_P61626.cif already exists
[7/73] Ribonuclease A (P61823) — 150 aa
  [skip] OF3_P61823.cif already exists
[8/73] BPTI (aprotinin) (P00974) — 100 aa
  [skip] OF3_P00974.cif already exists
[9/73] HIV-1 protease (P03366) — 1447 aa
  [skip] OF3_P03366.cif already exists
[10/73] Dihydrofolate reductase (E. coli) (P0ABQ4) — 159 aa
  [skip] OF3_P0ABQ4.cif already exists
[11/73] Thioredoxin (E. coli) (P0AA25) — 109 aa
  [skip] OF3_P0AA25.cif already exists
[12/73] CheY response regulator (P0AE67) — 129 aa
  [skip] OF3_P0AE67.cif already ex

,protein_name,uniprot_id,path,status
0,Barnase,P00648,data/openfold3/OF3_P00648.cif,ok
1,Protein G (GB1 domain parent),P06654,data/openfold3/OF3_P06654.cif,ok
2,Adenylate kinase (E. coli),P69441,data/openfold3/OF3_P69441.cif,ok
3,T4 lysozyme,P00720,data/openfold3/OF3_P00720.cif,ok
4,Myoglobin,P02185,data/openfold3/OF3_P02185.cif,ok
...,...,...,...,...
68,Nucleoporin Nup133,Q8WUM0,data/openfold3/OF3_Q8WUM0.cif,ok
69,Importin beta-1,Q14974,data/openfold3/OF3_Q14974.cif,ok
70,V-ATPase subunit A,P38606,data/openfold3/OF3_P38606.cif,ok
71,Photosystem II protein D1,P0C0S1,data/openfold3/OF3_P0C0S1.cif,ok
